# Train LightGBM v1 cho Steam Hybrid Recommendation trên Google Colab

Notebook này dùng để train LightGBM v1 từ dữ liệu Parquet đã export ra GCS.

Thiết kế chính:
- Không dùng `pd.read_parquet()` full bảng nếu dễ tràn RAM.
- Đọc Parquet theo batch bằng PyArrow.
- Ghi feature vào `numpy.memmap` trên ổ đĩa Colab.
- Train LightGBM CPU.
- Lưu model, metrics, feature importance vào Google Drive.

Bạn chỉ cần sửa `GCS_PREFIX` nếu folder export khác.

In [ ]:
# =========================================
# 1. Install libraries
# =========================================
!pip install -q lightgbm pyarrow gcsfs scikit-learn pandas matplotlib

In [ ]:
# =========================================
# 2. Authenticate Google Cloud
# =========================================
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "project-79499e5c-69d7-42b8-864"
BUCKET = "truong_bigdata_24032026_init"

print("PROJECT_ID:", PROJECT_ID)
print("BUCKET:", BUCKET)

PROJECT_ID: project-79499e5c-69d7-42b8-864
BUCKET: truong_bigdata_24032026_init


In [ ]:
# =========================================
# 3. Mount Google Drive for output
# =========================================
from google.colab import drive
drive.mount("/content/drive")

import os

DRIVE_OUT_DIR = "/content/drive/MyDrive/Steam_Hybrid_Recommender_Demo"
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

print("Output folder:", DRIVE_OUT_DIR)

Mounted at /content/drive
Output folder: /content/drive/MyDrive/Steam_Hybrid_Recommender_Demo


In [ ]:
# =========================================
# 4. Imports and GCS path config
# =========================================
import os
import gc
import json
import numpy as np
import pandas as pd
import gcsfs
import pyarrow.parquet as pq
import lightgbm as lgb
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss

# Folder chứa các file parquet numeric-only đã export từ BigQuery
# Không thêm gs:// ở đầu trong GCS_PREFIX vì gcsfs dùng dạng bucket/path
GCS_PREFIX = "truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/"

# Nếu folder của bạn khác, sửa dòng trên.
print("GCS_PREFIX:", GCS_PREFIX)

GCS_PREFIX: truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/


In [ ]:
# =========================================
# 5. Feature columns
# =========================================

# ID cols chỉ cần nếu bạn muốn đọc lại để predict/rank sau train.
# Khi train memmap để tránh RAM, ta KHÔNG đọc id_cols.
id_cols = ["user_id", "app_id", "user_idx", "app_idx"]
target_col = "label"

# Bộ feature basic, đủ mạnh để train v1 và nhẹ RAM hơn.
feature_cols = [
    "als_score",
    "als_rank",
    "als_inverse_rank",
    "als_rank_log",

    "language_score",
    "has_language_match",
    "matched_language_count",

    "user_activity_score",
    "user_weight",
    "user_reviews_log",
    "user_products_log",

    "platform_score",
    "steam_deck_score",

    "positive_ratio",
    "positive_review_share",
    "review_count_final",
    "avg_recommendation_hours",

    "price_final_clean",
    "discount",
    "is_free",

    "quality_score",
    "popularity_score",
    "engagement_score",
    "price_attractiveness_score",
    "recency_score",
    "game_profile_score",

    "als_x_language_score",
    "als_x_popularity_score",
    "als_x_quality_score",
    "user_activity_x_game_engagement",
]

all_cols = [target_col] + feature_cols

print("Number of features:", len(feature_cols))
print(feature_cols)

Number of features: 30
['als_score', 'als_rank', 'als_inverse_rank', 'als_rank_log', 'language_score', 'has_language_match', 'matched_language_count', 'user_activity_score', 'user_weight', 'user_reviews_log', 'user_products_log', 'platform_score', 'steam_deck_score', 'positive_ratio', 'positive_review_share', 'review_count_final', 'avg_recommendation_hours', 'price_final_clean', 'discount', 'is_free', 'quality_score', 'popularity_score', 'engagement_score', 'price_attractiveness_score', 'recency_score', 'game_profile_score', 'als_x_language_score', 'als_x_popularity_score', 'als_x_quality_score', 'user_activity_x_game_engagement']


In [ ]:
# =========================================
# 6. List parquet files on GCS
# =========================================
fs = gcsfs.GCSFileSystem(project=PROJECT_ID)

parquet_files = fs.glob(GCS_PREFIX + "*.parquet")
if len(parquet_files) == 0:
    parquet_files = fs.glob(GCS_PREFIX + "**/*.parquet")

print("Found parquet files:", len(parquet_files))
for f in parquet_files[:10]:
    print(f)

if len(parquet_files) == 0:
    raise FileNotFoundError(f"No parquet files found under gs://{GCS_PREFIX}")

Found parquet files: 167
truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/000000000000.parquet
truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/000000000001.parquet
truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/000000000002.parquet
truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/000000000003.parquet
truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/000000000004.parquet
truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/000000000005.parquet
truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/000000000006.parquet
truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/000000000007.parquet
truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/000000000008.parquet
truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/000000000009.parquet


In [ ]:
# =========================================
# 7. Check schema of first parquet file
# =========================================
with fs.open(parquet_files[0], "rb") as file_obj:
    pf = pq.ParquetFile(file_obj)
    print("Rows in first file:", pf.metadata.num_rows)
    print("Columns in first file:")
    print(pf.schema)

# Check missing expected columns
with fs.open(parquet_files[0], "rb") as file_obj:
    pf = pq.ParquetFile(file_obj)
    available_cols = set(pf.schema.names)

missing_cols = [c for c in all_cols if c not in available_cols]
print("Missing cols:", missing_cols)

if missing_cols:
    raise ValueError("Some required columns are missing. Check export SQL or feature_cols list.")

Rows in first file: 100800
Columns in first file:
required group field_id=-1 schema {
  optional int64 field_id=-1 user_id;
  optional int64 field_id=-1 app_id;
  optional int64 field_id=-1 user_idx;
  optional int64 field_id=-1 app_idx;
  optional int64 field_id=-1 label;
  optional double field_id=-1 als_score;
  optional int64 field_id=-1 als_rank;
  optional double field_id=-1 als_inverse_rank;
  optional double field_id=-1 als_rank_log;
  optional double field_id=-1 user_products_raw;
  optional double field_id=-1 user_reviews_raw;
  optional double field_id=-1 user_products_owned;
  optional double field_id=-1 user_reviews_count;
  optional double field_id=-1 user_products_log;
  optional double field_id=-1 user_reviews_log;
  optional double field_id=-1 user_review_density;
  optional double field_id=-1 user_activity_score;
  optional double field_id=-1 user_weight;
  optional int64 field_id=-1 user_has_enough_signal;
  optional int64 field_id=-1 user_is_outlier;
  optional int6

In [ ]:
# =========================================
# 8. Count total rows without loading full data
# =========================================
total_rows = 0
for i, f in enumerate(parquet_files, start=1):
    with fs.open(f, "rb") as file_obj:
        pf = pq.ParquetFile(file_obj)
        total_rows += pf.metadata.num_rows
    if i % 20 == 0:
        print(f"Checked {i}/{len(parquet_files)} files, rows so far: {total_rows:,}")

n_rows = total_rows
n_features = len(feature_cols)

print("Total rows:", f"{n_rows:,}")
print("Number of features:", n_features)

Checked 20/167 files, rows so far: 2,017,286
Checked 40/167 files, rows so far: 4,034,123
Checked 60/167 files, rows so far: 6,051,770
Checked 80/167 files, rows so far: 8,069,185
Checked 100/167 files, rows so far: 10,086,619
Checked 120/167 files, rows so far: 12,106,755
Checked 140/167 files, rows so far: 14,123,852
Checked 160/167 files, rows so far: 16,140,767
Total rows: 16,846,271
Number of features: 30


In [ ]:
# =========================================
# 9. Optional: quick sample read to verify data
# =========================================
with fs.open(parquet_files[0], "rb") as file_obj:
    pf = pq.ParquetFile(file_obj)
    batch = next(pf.iter_batches(batch_size=5, columns=all_cols))
    sample_df = batch.to_pandas()

display(sample_df)
print(sample_df.dtypes)

,label,als_score,als_rank,als_inverse_rank,als_rank_log,language_score,has_language_match,matched_language_count,user_activity_score,user_weight,...,quality_score,popularity_score,engagement_score,price_attractiveness_score,recency_score,game_profile_score,als_x_language_score,als_x_popularity_score,als_x_quality_score,user_activity_x_game_engagement
0,0,0.427105,45,0.022222,3.828641,0.331426,1,1,0.0,1.0,...,0.3395,0.050172,0.022577,0.6375,0.1,0.29153,0.141554,0.021429,0.145002,0.0
1,0,0.429173,37,0.027027,3.637586,0.331426,1,1,0.0,1.0,...,0.3395,0.050172,0.022577,0.6375,0.1,0.29153,0.142239,0.021532,0.145704,0.0
2,0,0.451150,35,0.028571,3.583519,0.331426,1,1,0.0,1.0,...,0.3395,0.050172,0.022577,0.6375,0.1,0.29153,0.149523,0.022635,0.153165,0.0
3,0,0.745587,16,0.062500,2.833213,0.331426,1,1,0.0,1.0,...,0.3395,0.050172,0.022577,0.6375,0.1,0.29153,0.247107,0.037407,0.253127,0.0
4,0,0.638949,32,0.031250,3.496508,0.331426,1,1,0.0,1.0,...,0.3395,0.050172,0.022577,0.6375,0.1,0.29153,0.211764,0.032057,0.216923,0.0


label                                int64
als_score                          float64
als_rank                             int64
als_inverse_rank                   float64
als_rank_log                       float64
language_score                     float64
has_language_match                   int64
matched_language_count               int64
user_activity_score                float64
user_weight                        float64
user_reviews_log                   float64
user_products_log                  float64
platform_score                     float64
steam_deck_score                   float64
positive_ratio                     float64
positive_review_share              float64
review_count_final                 float64
avg_recommendation_hours           float64
price_final_clean                  float64
discount                           float64
is_free                              int64
quality_score                      float64
popularity_score                   float64
engagement_

In [ ]:
# =========================================
# 10. Create numpy memmap files on Colab disk
# =========================================
MEMMAP_DIR = "/content/lgbm_memmap"
os.makedirs(MEMMAP_DIR, exist_ok=True)

X_path = f"{MEMMAP_DIR}/X_float32.dat"
y_path = f"{MEMMAP_DIR}/y_int8.dat"

# X: float32 feature matrix
# y: int8 label vector
X_mm = np.memmap(
    X_path,
    dtype="float32",
    mode="w+",
    shape=(n_rows, n_features)
)

y_mm = np.memmap(
    y_path,
    dtype="int8",
    mode="w+",
    shape=(n_rows,)
)

print("Memmap created:")
print("X:", X_mm.shape, X_mm.dtype)
print("y:", y_mm.shape, y_mm.dtype)
print("X file:", X_path)
print("y file:", y_path)

Memmap created:
X: (16846271, 30) float32
y: (16846271,) int8
X file: /content/lgbm_memmap/X_float32.dat
y file: /content/lgbm_memmap/y_int8.dat


In [ ]:
# =========================================
# 11. Read parquet by batches and write into memmap
# =========================================
# Nếu RAM vẫn căng, giảm batch_size xuống 100_000 hoặc 50_000
batch_size = 100_000
row_cursor = 0

for file_idx, f in enumerate(parquet_files, start=1):
    print(f"\n[{file_idx}/{len(parquet_files)}] Reading: {f}")

    with fs.open(f, "rb") as file_obj:
        pf = pq.ParquetFile(file_obj)

        for batch in pf.iter_batches(batch_size=batch_size, columns=all_cols):
            pdf = batch.to_pandas()

            # Label
            y_batch = (
                pd.to_numeric(pdf[target_col], errors="coerce")
                .fillna(0)
                .astype("int8")
                .to_numpy()
            )

            # Feature matrix: convert each column to float32, fill NaN/inf
            X_cols = []
            for c in feature_cols:
                arr = pd.to_numeric(pdf[c], errors="coerce").astype("float32").to_numpy()
                arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0).astype("float32")
                X_cols.append(arr)

            X_batch = np.column_stack(X_cols).astype("float32")
            n = len(pdf)

            X_mm[row_cursor:row_cursor+n, :] = X_batch
            y_mm[row_cursor:row_cursor+n] = y_batch

            row_cursor += n

            del pdf, X_batch, X_cols, y_batch, batch
            gc.collect()

    X_mm.flush()
    y_mm.flush()
    print("Rows loaded so far:", f"{row_cursor:,}")

print("\nDone.")
print("Total loaded rows:", f"{row_cursor:,}")

if row_cursor != n_rows:
    print("WARNING: loaded row count differs from metadata row count.")


[1/167] Reading: truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/000000000000.parquet
Rows loaded so far: 100,800

[2/167] Reading: truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/000000000001.parquet
Rows loaded so far: 201,513

[3/167] Reading: truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/000000000002.parquet
Rows loaded so far: 302,656

[4/167] Reading: truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/000000000003.parquet
Rows loaded so far: 403,430

[5/167] Reading: truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/000000000004.parquet
Rows loaded so far: 504,110

[6/167] Reading: truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/000000000005.parquet
Rows loaded so far: 605,342

[7/167] Reading: truong_bigdata_24032026_init/steam/gold/lgbm_train_features_v1_16m_numeric/000000000006.parquet
Rows loaded so far: 706,425

[8/16

In [ ]:
# =========================================
# 12. Reopen memmap as read-only and inspect label distribution
# =========================================
X = np.memmap(
    X_path,
    dtype="float32",
    mode="r",
    shape=(n_rows, n_features)
)

y = np.memmap(
    y_path,
    dtype="int8",
    mode="r",
    shape=(n_rows,)
)

print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)
print("Positive rows:", int(y.sum()))
print("Negative rows:", int(len(y) - y.sum()))
print("Positive rate:", float(y.mean()))

X: (16846271, 30) float32
y: (16846271,) int8
Positive rows: 1425281
Negative rows: 15420990
Positive rate: 0.08460513308850368


In [ ]:
# =========================================
# 13. Create train/valid split
# =========================================
# Lưu ý: np.asarray(y) tạo vector label trong RAM, nhưng y chỉ int8 nên ~16MB cho 16M rows, ổn.
indices = np.arange(n_rows)

train_idx, valid_idx = train_test_split(
    indices,
    test_size=0.15,
    random_state=42,
    stratify=np.asarray(y)
)

print("Train rows:", f"{len(train_idx):,}")
print("Valid rows:", f"{len(valid_idx):,}")

Train rows: 14,319,330
Valid rows: 2,526,941


In [ ]:
# =========================================
# 14. Materialize train/valid arrays
# =========================================
# Bước này copy X_train, X_valid vào RAM.
# Với 16M x 30 float32, thường khoảng ~1.9GB trước overhead, có thể chấp nhận trên Colab RAM cao.
# Nếu vẫn tràn RAM, xem cell fallback phía dưới.

try:
    X_train = X[train_idx]
    y_train = y[train_idx]

    X_valid = X[valid_idx]
    y_valid = y[valid_idx]

    print("X_train:", X_train.shape, X_train.dtype)
    print("X_valid:", X_valid.shape, X_valid.dtype)
    print("Train positive rate:", float(y_train.mean()))
    print("Valid positive rate:", float(y_valid.mean()))
except MemoryError:
    print("MemoryError: Không đủ RAM để materialize train/valid.")
    raise

X_train: (14319330, 30) float32
X_valid: (2526941, 30) float32
Train positive rate: 0.08460514563181377
Valid positive rate: 0.08460506200975805


In [ ]:
# =========================================
# 15. Train LightGBM CPU - fixed version
# =========================================
import lightgbm as lgb
import numpy as np
import pandas as pd
import gc
import os

num_pos = int(y_train.sum())
num_neg = int(len(y_train) - num_pos)
scale_pos_weight = num_neg / max(num_pos, 1)

print("num_pos:", f"{num_pos:,}")
print("num_neg:", f"{num_neg:,}")
print("scale_pos_weight:", scale_pos_weight)

# Ép kiểu để giảm RAM
X_train = X_train.astype(np.float32)
X_valid = X_valid.astype(np.float32)
y_train = y_train.astype(np.int8)
y_valid = y_valid.astype(np.int8)

# Kiểm tra nhanh NaN / Inf
print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)

print("Train positive rate:", y_train.mean())
print("Valid positive rate:", y_valid.mean())

params = {
    "objective": "binary",

    # Đặt average_precision lên trước để early stopping ưu tiên PR-AUC
    # Vì bài recommend/ranking lệch nhãn, AP quan trọng hơn logloss
    "metric": ["average_precision", "auc"],

    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": -1,
    "min_data_in_leaf": 200,

    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,

    "lambda_l1": 0.0,
    "lambda_l2": 1.0,

    # Không hard-code, dùng đúng tỷ lệ train hiện tại
    "scale_pos_weight": scale_pos_weight,

    # Tối ưu cho CPU / Colab
    "force_col_wise": True,
    "num_threads": os.cpu_count(),

    "verbosity": -1,
    "seed": 42,
    "feature_fraction_seed": 42,
    "bagging_seed": 42,
    "data_random_seed": 42,
}

train_data = lgb.Dataset(
    X_train,
    label=y_train,
    feature_name=feature_cols,
    free_raw_data=True
)

valid_data = lgb.Dataset(
    X_valid,
    label=y_valid,
    feature_name=feature_cols,
    reference=train_data,
    free_raw_data=True
)

model = lgb.train(
    params,
    train_data,
    valid_sets=[valid_data],
    valid_names=["valid"],
    num_boost_round=500,
    callbacks=[
        # Quan trọng: chỉ dùng metric đầu tiên = average_precision để early stopping
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=20),
    ],
)

print("=" * 80)
print("Best iteration:", model.best_iteration)
print("Best score:", model.best_score)
print("=" * 80)

gc.collect()

num_pos: 1,211,489
num_neg: 13,107,841
scale_pos_weight: 10.819612064162365
X_train shape: (14319330, 30)
X_valid shape: (2526941, 30)
Train positive rate: 0.08460514563181377
Valid positive rate: 0.08460506200975805
Training until validation scores don't improve for 50 rounds
[20]	valid's average_precision: 0.205171	valid's auc: 0.711096
[40]	valid's average_precision: 0.207093	valid's auc: 0.713588
[60]	valid's average_precision: 0.208599	valid's auc: 0.715862
[80]	valid's average_precision: 0.210782	valid's auc: 0.718116
[100]	valid's average_precision: 0.212373	valid's auc: 0.72024
[120]	valid's average_precision: 0.213739	valid's auc: 0.721992
[140]	valid's average_precision: 0.215053	valid's auc: 0.723652
[160]	valid's average_precision: 0.21625	valid's auc: 0.724834
[180]	valid's average_precision: 0.217397	valid's auc: 0.725789
[200]	valid's average_precision: 0.218335	valid's auc: 0.726565
[220]	valid's average_precision: 0.219138	valid's auc: 0.727199
[240]	valid's average_pr

453

In [ ]:
# =========================================
# 16. Evaluation metrics
# =========================================
valid_pred = model.predict(X_valid, num_iteration=model.best_iteration)

auc = roc_auc_score(y_valid, valid_pred)
ap = average_precision_score(y_valid, valid_pred)
ll = log_loss(y_valid, valid_pred)

metrics_df = pd.DataFrame([{
    "model": "lightgbm_v1_16m_colab_cpu_memmap",
    "rows": int(n_rows),
    "features": int(len(feature_cols)),
    "best_iteration": int(model.best_iteration),
    "valid_auc": float(auc),
    "valid_average_precision": float(ap),
    "valid_logloss": float(ll),
    "positive_rate": float(y.mean()),
    "scale_pos_weight": float(scale_pos_weight),
}])

display(metrics_df)

metrics_path = f"{DRIVE_OUT_DIR}/metrics_lgbm_v1_16m_colab_cpu_memmap.csv"
metrics_df.to_csv(metrics_path, index=False)

print("Saved metrics:", metrics_path)

,model,rows,features,best_iteration,valid_auc,valid_average_precision,valid_logloss,positive_rate,scale_pos_weight
0,lightgbm_v1_16m_colab_cpu_memmap,16846271,30,500,0.731853,0.223341,0.605243,0.084605,10.819612


Saved metrics: /content/drive/MyDrive/Steam_Hybrid_Recommender_Demo/metrics_lgbm_v1_16m_colab_cpu_memmap.csv


In [ ]:
# =========================================
# 17. Save model and feature importance
# =========================================
model_path = f"{DRIVE_OUT_DIR}/lgbm_v1_16m_colab_cpu_memmap_model.txt"
model.save_model(model_path)

print("Saved model:", model_path)

fi = pd.DataFrame({
    "feature": feature_cols,
    "importance_gain": model.feature_importance(importance_type="gain"),
    "importance_split": model.feature_importance(importance_type="split"),
}).sort_values("importance_gain", ascending=False)

fi_path = f"{DRIVE_OUT_DIR}/feature_importance_lgbm_v1_16m_colab_cpu_memmap.csv"
fi.to_csv(fi_path, index=False)

display(fi.head(30))
print("Saved feature importance:", fi_path)

Saved model: /content/drive/MyDrive/Steam_Hybrid_Recommender_Demo/lgbm_v1_16m_colab_cpu_memmap_model.txt


,feature,importance_gain,importance_split
0,als_score,1.580729e+07,765
27,als_x_popularity_score,1.453056e+07,556
28,als_x_quality_score,7.243105e+06,479
25,game_profile_score,3.606365e+06,2613
24,recency_score,3.228059e+06,2397
4,language_score,3.034466e+06,2451
17,price_final_clean,2.258011e+06,1498
13,positive_ratio,1.608950e+06,1540
1,als_rank,1.470121e+06,850
26,als_x_language_score,4.927767e+05,427


Saved feature importance: /content/drive/MyDrive/Steam_Hybrid_Recommender_Demo/feature_importance_lgbm_v1_16m_colab_cpu_memmap.csv


In [ ]:
# =========================================
# 16. Predict validation
# =========================================
import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

valid_pred = model.predict(
    X_valid,
    num_iteration=model.best_iteration
)

print("valid_pred min:", valid_pred.min())
print("valid_pred max:", valid_pred.max())
print("valid_pred mean:", valid_pred.mean())
print("valid_pred std:", valid_pred.std())

valid_pred min: 0.02295248958027875
valid_pred max: 0.934033628208657
valid_pred mean: 0.43490372671450306
valid_pred std: 0.178561623502728
